# Original vs reconstruction: M20, Gini, C, A
The goal of this notebook is to reproduce **Figure 4** of
[Euclid Collaboration: Csizi et al. (2025), *Euclid preparation. LXVII. Deep learning true galaxy
morphologies for weak lensing shear bias calibration*, A&A 695, A283](https://doi.org/10.1051/0004-6361/202452129)
([arXiv:2409.07528](https://arxiv.org/abs/2409.07528)): a galaxy-by-galaxy comparison of morphological
proxies measured on the original images (x-axis) and on their reconstructions by the generative
model (y-axis).

It reads the FITS catalogues written by `run_morphometrics.py`
(`catalog_real.fits` and `catalog_reconstruction.fits`) and draws panels (a)–(d): $M_{20}$, Gini $G$,
concentration $C$ and asymmetry $A$. Smoothness $S$ (panel e) is not computed by this pipeline, and the
example galaxies of panel (f) are left out.

Unlike the paper's log-scaled density, the colour shows the **fraction of objects** per hexagon on a
linear scale, and black contours enclose **68 % and 95 %** of the objects. Both are independent of the
sample size, so the scatter is not visually inflated when the catalogue is large.

In [ ]:
# !pip install -q astropy   # usually already installed on Colab

import os

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

In [ ]:
# --- Load the FITS catalogues ------------------------------------------------
# Option A (Colab): upload both files by hand       -> USE_UPLOAD = True
# Option B (Colab): read them from Google Drive     -> USE_DRIVE = True and set DATA_DIR below
# Option C (local): set DATA_DIR to the folder holding the catalogues (Colab cannot see local paths)
USE_UPLOAD = True
USE_DRIVE = False
DATA_DIR = "."

REAL_FILE = "catalog_real.fits"
RECO_FILE = "catalog_reconstruction.fits"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_UPLOAD:
    from google.colab import files
    files.upload()  # select catalog_real.fits and catalog_reconstruction.fits
    DATA_DIR = "."
elif IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/results_droppedDB_newflow"

real = Table.read(os.path.join(DATA_DIR, REAL_FILE))
reco = Table.read(os.path.join(DATA_DIR, RECO_FILE))
print(len(real), "original objects,", len(reco), "reconstructions")

In [ ]:
# --- Matching and selection --------------------------------------------------
# Both catalogues are built from the same stamps: objects are matched by IDENT.
if not np.array_equal(np.asarray(real["IDENT"]), np.asarray(reco["IDENT"])):
    order = np.argsort(np.asarray(reco["IDENT"]))
    idx = order[np.searchsorted(np.asarray(reco["IDENT"])[order], np.asarray(real["IDENT"]))]
    reco = reco[idx]
    assert np.array_equal(np.asarray(real["IDENT"]), np.asarray(reco["IDENT"]))

# An object is kept if the (R) morphology measurement succeeded on BOTH the original and the reconstruction.
# -9 is the sentinel value written by the pipeline when a measurement fails.
base_mask = np.asarray(real["flag_morph"]) & np.asarray(reco["flag_morph"])


def valid(col):
    x = np.asarray(real[col], dtype=float)
    y = np.asarray(reco[col], dtype=float)
    m = base_mask & np.isfinite(x) & np.isfinite(y) & (x != -9) & (y != -9)
    return x[m], y[m]

In [ ]:
# --- Panel settings ----------------------------------------------------------
# limits=None -> automatic limits (0.5-99.5 percentiles of both samples)
# Limits used in the paper: M20 (-2.5, -0.5), Gini (0, 0.65), C (1, 4), A (-0.05, 0.4)
PANELS = [
    dict(col="M20",  label=r"(a) $M_{20}$ coefficient",  limits=None),
    dict(col="Gini", label=r"(b) Gini coefficient $G$",  limits=None),
    dict(col="C",    label=r"(c) Concentration $C$",     limits=None),
    dict(col="A",    label=r"(d) Asymmetry $A$",         limits=None),
]
XLABEL = "original"
YLABEL = "model reconstruction"
GRIDSIZE = 60
HIST_BINS = 30
CONTOUR_FRACTIONS = (0.95, 0.68)  # fraction of objects enclosed by each contour
CONTOUR_BINS = 40


def auto_limits(x, y, lo=0.5, hi=99.5, pad=0.05):
    v = np.concatenate([x, y])
    a, b = np.percentile(v, [lo, hi])
    d = (b - a) * pad
    return a - d, b + d


def enclosed_levels(H, fractions):
    """Histogram levels whose contours enclose the given fractions of objects."""
    h = np.sort(H.ravel())[::-1]
    cum = np.cumsum(h) / h.sum()
    return sorted(h[np.searchsorted(cum, f)] for f in fractions)


def scatter_panel(ax, x, y, label, limits):
    lim = limits if limits is not None else auto_limits(x, y)
    in_box = (x >= lim[0]) & (x <= lim[1]) & (y >= lim[0]) & (y <= lim[1])
    xs, ys = x[in_box], y[in_box]

    # Colour = fraction of objects per hexagon (linear, independent of sample size)
    w = np.full(len(xs), 1.0 / len(xs))
    hb = ax.hexbin(xs, ys, C=w, reduce_C_function=np.sum, gridsize=GRIDSIZE,
                   extent=(*lim, *lim), cmap="Blues", mincnt=1, linewidths=0)
    hb.set_clim(0, np.percentile(np.ma.compressed(hb.get_array()), 99.5))

    # Contours enclosing 68 % and 95 % of the objects
    H, xe, ye = np.histogram2d(xs, ys, bins=CONTOUR_BINS, range=[lim, lim])
    xc, yc = 0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1])
    ax.contour(xc, yc, H.T, levels=enclosed_levels(H, CONTOUR_FRACTIONS),
               colors="k", linewidths=[0.8, 1.3])

    ax.plot(lim, lim, ls=":", color="k", lw=1.2)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal")
    ax.text(0.04, 0.95, label, transform=ax.transAxes, va="top", ha="left", fontsize=12)
    ax.set_xlabel(XLABEL, fontsize=12)
    ax.set_ylabel(YLABEL, fontsize=12)
    ax.tick_params(direction="in", top=True, right=True)

    # Marginal histograms: original on top, reconstruction on the right
    bins = np.linspace(*lim, HIST_BINS + 1)
    ax_top = ax.inset_axes([0, 1.0, 1, 0.15], sharex=ax)
    ax_top.hist(x[(x >= lim[0]) & (x <= lim[1])], bins=bins, color="steelblue", alpha=0.8)
    ax_right = ax.inset_axes([1.0, 0, 0.15, 1], sharey=ax)
    ax_right.hist(y[(y >= lim[0]) & (y <= lim[1])], bins=bins, color="steelblue", alpha=0.8,
                  orientation="horizontal")
    for a in (ax_top, ax_right):
        a.axis("off")

    # Summary statistics (computed on all valid objects, not only those inside the plot limits)
    r = np.corrcoef(x, y)[0, 1]
    diff = y - x
    bias = np.median(diff)
    nmad = 1.4826 * np.median(np.abs(diff - bias))
    return dict(n=len(x), pearson_r=r, median_bias=bias, nmad=nmad)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
summary = {}
for ax, p in zip(axes.ravel(), PANELS):
    x, y = valid(p["col"])
    summary[p["col"]] = scatter_panel(ax, x, y, p["label"], p["limits"])
fig.subplots_adjust(wspace=0.45, hspace=0.4)
plt.show()

print(f"{'':6s} {'N':>7s} {'r':>7s} {'med. bias':>11s} {'NMAD':>8s}")
for col, s in summary.items():
    print(f"{col:6s} {s['n']:7d} {s['pearson_r']:7.3f} {s['median_bias']:+11.4f} {s['nmad']:8.4f}")

In [ ]:
# fig.savefig("real_vs_reconstruction_morphology.pdf", bbox_inches="tight")

---

## Noise control: is the comparison in panels (a)–(d) actually fair?

Panels (a)–(d) above compare morphology measured on real images against morphology measured on
autoencoder reconstructions. That comparison is only meaningful if the two sets of images carry
**the same amount of noise**, because every indicator plotted there depends on it:

- `segmap.R` thresholds on quantiles of the image's own pixel values, so the segmentation
  — and with it `size`, Gini, `M20`, `M`, `I`, `D` — moves with the noise level;
- `A_statistic` subtracts a background-asymmetry term estimated from pixels outside the
  segmentation map, so `A` is biased if that background is not statistically the same;
- `sn` itself is source brightness divided by the background scatter of the same image.

Reconstructions come out (nearly) noise-free, so `run_morphometrics.py` gives them a synthetic
noise realization — `galmorph.data.add_noise`, white Gaussian noise scaled by the dataset's
per-pixel noise map. The cells below check whether that worked.

**What the noise map is.** These noise maps come from Euclid Q1 MER mosaics. Per the
[MER DPDD](https://euclid.esac.esa.int/dr/q1/dpdd/merdpd/dpcards/mer_bksmosaic.html), the
`RmsStorage` product is "the propagated rms associated to the mosaic" — a per-pixel **standard
deviation**, in the same units as the science mosaic (ADU/s for VIS). So `add_noise`'s
`images + N(0,1) * noise_map` uses the right power of σ. The same document adds a caveat that
matters here: *"the VIS and NIR rms values contain a Poisson noise contribution that could be
significant for bright sources"* — the map is **not** background-only, it rises where the
galaxy is bright.

Three checks follow: **(e)** whether the noise came out at the right amplitude, using `sn`;
**(f)** how to read that; and **(g)** an optional image-level test of the noise map itself.


In [ ]:
# --- (e) Noise control: signal-to-noise comparison ---------------------------
# `sn` is computed by the R pipeline separately for each image, from that
# image alone (galmorph/r_indicators/interface.R, lines 43-59):
#
#     w     <- which(smap.all == 0)        # pixels outside every detection
#     muhat <- mean(img[w], trim = 0.05)   # background level
#     sdhat <- sd(img[w])                  # background scatter, IN THIS IMAGE
#     sn    <- median((img[central] - muhat) / sdhat)   # over segmap pixels
#                                                       # within 0.5 * size
#
# It never reads the noise map, which is what makes it an independent probe of
# how much noise actually ended up in the reconstructions.
#
# CAREFUL: sn is a surface brightness divided by a noise level, so THREE things
# move it, and only the third is about noise:
#
#     sn  ~  SB_central / sigma_noise ,   SB_central ~ flux / size^2
#
# An autoencoder that loses flux, or that blurs the galaxy, lowers sn without
# anything being wrong with the noise. Both are measurable here from the HSM
# columns (`amp` = best-fit Gaussian total flux, `sigma_e` = its width), so we
# divide them out and isolate the noise term:
#
#     sigma_reco / sigma_real  =  R_sn * R_amp / R_size^2
#
# with R_sn = median(sn_real)/median(sn_reco), R_amp = median(amp_reco/amp_real)
# and R_size = median(sigma_e_reco/sigma_e_real).

if "sn" not in real.colnames or "sn" not in reco.colnames:
    print("No `sn` column in the catalogues -- was run_morphometrics.py run with --skip-r?")
else:
    sn_x, sn_y = valid("sn")
    R_sn = np.median(sn_x) / np.median(sn_y)

    # Moments columns carry their own flag, independent of flag_morph.
    def _moment_ratio(col):
        x = np.asarray(real[col], float)
        y = np.asarray(reco[col], float)
        m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
        if "flag_moments" in real.colnames and "flag_moments" in reco.colnames:
            m &= np.asarray(real["flag_moments"]) & np.asarray(reco["flag_moments"])
        return (np.median(y[m] / x[m]), int(m.sum())) if m.any() else (None, 0)

    R_amp,  n_amp  = _moment_ratio("amp")      if "amp" in real.colnames else (None, 0)
    R_size, n_size = _moment_ratio("sigma_e")  if "sigma_e" in real.colnames else (None, 0)

    lim = (min(np.percentile(sn_x, 0.5), np.percentile(sn_y, 0.5)),
           max(np.percentile(sn_x, 99.5), np.percentile(sn_y, 99.5)))
    bins = np.linspace(*lim, 60)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

    axes[0].hist(sn_x, bins=bins, histtype="step", lw=2, label="original", color="steelblue")
    axes[0].hist(sn_y, bins=bins, histtype="step", lw=2, label="reconstruction", color="crimson")
    axes[0].axvline(np.median(sn_x), color="steelblue", ls=":", lw=1.5)
    axes[0].axvline(np.median(sn_y), color="crimson", ls=":", lw=1.5)
    axes[0].set_xlabel("$S/N$ (median central pixel significance)")
    axes[0].set_ylabel("objects")
    axes[0].legend()
    axes[0].set_title("(e) S/N distributions")

    in_box = (sn_x >= lim[0]) & (sn_x <= lim[1]) & (sn_y >= lim[0]) & (sn_y <= lim[1])
    hb = axes[1].hexbin(sn_x[in_box], sn_y[in_box], gridsize=50,
                        extent=(*lim, *lim), cmap="Blues", mincnt=1, linewidths=0)
    hb.set_clim(0, np.percentile(np.ma.compressed(hb.get_array()), 99.5))
    axes[1].plot(lim, lim, ls=":", color="k", lw=1.2)
    axes[1].set_xlim(lim); axes[1].set_ylim(lim); axes[1].set_aspect("equal")
    axes[1].set_xlabel("$S/N$ original")
    axes[1].set_ylabel("$S/N$ reconstruction")
    axes[1].set_title("per-object")
    plt.tight_layout(); plt.show()

    print(f"N objects                        : {len(sn_x)}")
    print(f"median S/N  original             : {np.median(sn_x):.3f}")
    print(f"median S/N  reconstruction       : {np.median(sn_y):.3f}")
    print(f"R_sn    = sn_real / sn_reco      : {R_sn:.4f}")

    corrected = R_sn
    if R_amp is not None:
        print(f"R_amp   = flux   reco / real     : {R_amp:.4f}  ({100*(R_amp-1):+.2f} %)  [N={n_amp}]")
        corrected *= R_amp
    if R_size is not None:
        print(f"R_size  = size   reco / real     : {R_size:.4f}  ({100*(R_size-1):+.2f} %)  [N={n_size}]")
        corrected /= R_size ** 2
    if R_amp is not None and R_size is not None:
        print(f"  surface brightness reco/real   : {R_amp/R_size**2:.4f}  ({100*(R_amp/R_size**2-1):+.2f} %)")

    print(f"\nsigma_reco / sigma_real          : {corrected:.4f}  "
          f"({100*(corrected-1):+.1f} % noise in the reconstructions)")

    if abs(corrected - 1) < 0.05:
        print("\nNoise is matched once flux and size are divided out: panels (a)-(d) can be")
        print("read as model error.")
    elif corrected > 1:
        excess_var = corrected ** 2 - 1
        print("\nReconstructions carry MORE noise than the originals, and it is not a flux or")
        print("blurring artefact -- those have been divided out. Excess variance is")
        print(f"{100*excess_var:.0f} % of sigma^2, i.e. a residual of {np.sqrt(excess_var):.2f} * sigma already")
        print("present in the reconstruction BEFORE add_noise, on top of which a full sigma")
        print("was then added. If cell (g) shows the noise map matches the real background,")
        print("this is autoencoder noise leakage, not a bad noise map: inject the deficit in")
        print(f"quadrature instead, sigma_add = sigma * sqrt(1 - {excess_var:.2f}) = {np.sqrt(max(1-excess_var,0)):.2f} * sigma.")
    else:
        print("\nReconstructions are CLEANER than the originals: the noise map is being")
        print("under-applied. Verify the column's convention (sigma vs variance vs weight)")
        print("with the cell below before trusting panels (a)-(d).")


### How to read the S/N panels

**The ratio of medians is the number to look at.** `sn` is inversely proportional to the
background scatter measured *inside each stamp*, so

$$\frac{\mathrm{median}(sn_{\rm real})}{\mathrm{median}(sn_{\rm reco})} \;\simeq\; \frac{\sigma_{\rm reco}}{\sigma_{\rm real}}$$

That ratio is a direct measurement of how much noise `galmorph.data.add_noise` actually put
into the reconstructions, relative to the real images. It needs no knowledge of the noise map
at all, which is exactly what makes it a useful cross-check on it.

| ratio | reading |
|---|---|
| ≈ 1.00 | the injected noise matches the real images. Panels (a)–(d) can be read as model error. |
| > 1.05 | reconstructions are **too noisy**. Likely causes: the reconstruction was not noise-free to begin with (the autoencoder passes some input noise through, and `add_noise` then adds a full σ on top), or the RMS map is inflated relative to the true pixel-to-pixel scatter. |
| < 0.95 | reconstructions are **too clean** — the noise map is being under-applied. Check the column's convention (σ vs variance vs inverse-variance weight). |

**A flux bias contaminates this ratio.** `sn` is a *ratio* of source brightness to background
scatter, so an autoencoder that systematically under-reconstructs flux also lowers `sn` without
any noise being wrong. Read this panel together with `flux_error.pdf` from
`run_morphometrics.py`: if the flux ratio is ≈ 1 but `sn` is off, the discrepancy is genuinely
in the noise.

**What this panel cannot tell you.** `sn` depends only on the *marginal* scatter σ, not on how
noise is distributed across spatial scales. White noise and spatially correlated noise with the
same σ give the same `sn`. Euclid mosaics are resampled (Lanczos3 for VIS), which correlates
neighbouring pixels, while `add_noise` injects strictly white noise. That difference is
invisible here but does affect `segmap`, and therefore Gini, `M20`, `A` and `M`. The optional
cell below tests it with a lag-1 autocorrelation.


In [ ]:
# --- (g) OPTIONAL: image-level check of the noise map itself -----------------
# This answers a different question than the `sn` plot above. `sn` tells you
# whether the noise came out right; this tells you whether the noise *map* is
# what the pipeline assumes it is. Needs the `galmorph` package and access to
# the dataset, so it is off by default.
#
# What Euclid Q1 actually provides (MER DPDD, DpdMer-BksMosaic / RmsStorage):
#   * "This FITS file contains the propagated rms associated to the mosaic."
#     -> a per-pixel STANDARD DEVIATION, in the same units as the science
#        mosaic (ADU/s for VIS). So `images + N(0,1)*noise_map` in
#        galmorph.data.add_noise uses the right power of sigma. Good.
#   * "Note that the VIS and NIR rms values contain a Poisson noise
#     contribution that could be significant for bright sources."
#     -> the map is NOT background-only: it rises where the galaxy is bright.
#
# That second point is why the amplitude test below is restricted to a BORDER
# strip: there the Poisson term is negligible, so the RMS map should match the
# background scatter measured directly in the stamp. Comparing over the whole
# stamp would fold in the source Poisson term and give a meaningless ratio.

RUN_IMAGE_CHECK = False        # <-- set to True to run

DATASET         = "your-org/your-dataset"   # <-- same values you pass to run_morphometrics.py
IMAGE_FIELD     = "sci_subtracted"
NOISE_MAP_FIELD = "noise_map"
SPLIT           = "train"
STAMP_SIZE      = 64
N_CHECK         = 300          # a few hundred stamps is plenty
BORDER          = 6            # width in pixels of the background strip

if RUN_IMAGE_CHECK:
    from galmorph.data import load_hf_stamps

    stamps, extra = load_hf_stamps(
        DATASET, split=SPLIT, image_field=IMAGE_FIELD,
        n_samples=N_CHECK, stamp_size=STAMP_SIZE,
        noise_map_field=NOISE_MAP_FIELD,
    )
    nmap = extra["noise_map"]

    # Background strip: the outer BORDER pixels on all four sides.
    edge = np.zeros((STAMP_SIZE, STAMP_SIZE), dtype=bool)
    edge[:BORDER, :] = edge[-BORDER:, :] = True
    edge[:, :BORDER] = edge[:, -BORDER:] = True

    # --- Test 1: amplitude. Measured scatter vs. what the noise map claims. ---
    # NMAD rather than std, so a stray source in the strip does not inflate it.
    def nmad(a, axis=None):
        med = np.median(a, axis=axis, keepdims=True)
        return 1.4826 * np.median(np.abs(a - med), axis=axis)

    sigma_measured = nmad(stamps[:, edge], axis=1)     # per stamp, from the pixels
    sigma_claimed  = np.median(nmap[:, edge], axis=1)  # per stamp, from the map
    ok = np.isfinite(sigma_measured) & np.isfinite(sigma_claimed) & (sigma_claimed > 0)
    k = np.median(sigma_measured[ok] / sigma_claimed[ok])

    # --- Test 2: correlation. White noise has lag-1 autocorrelation ~ 0. ------
    # Euclid mosaics are resampled (Lanczos3 for VIS), which correlates
    # neighbouring pixels. add_noise injects strictly WHITE noise, so if this
    # comes out clearly non-zero the two images differ in noise *structure*,
    # not just amplitude -- and `sn` is blind to that difference.
    top = stamps[:, :BORDER, :]                        # a clean horizontal strip
    a = top[:, :, :-1].ravel()
    b = top[:, :, 1:].ravel()
    rho1 = np.corrcoef(a, b)[0, 1]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(sigma_measured[ok] / sigma_claimed[ok], bins=50, color="steelblue")
    axes[0].axvline(1.0, color="k", ls=":", lw=1.5)
    axes[0].axvline(k, color="crimson", lw=1.5)
    axes[0].set_xlabel("measured background scatter / noise map")
    axes[0].set_ylabel("stamps")
    axes[0].set_title(f"amplitude: median ratio = {k:.3f}")

    axes[1].scatter(sigma_claimed[ok], sigma_measured[ok], s=4, alpha=0.3)
    lim = (0, np.percentile(sigma_claimed[ok], 99) * 1.1)
    axes[1].plot(lim, lim, ls=":", color="k")
    axes[1].set_xlim(lim); axes[1].set_ylim(lim)
    axes[1].set_xlabel("noise map (median over border)")
    axes[1].set_ylabel("measured scatter (NMAD over border)")
    axes[1].set_title(f"lag-1 autocorrelation = {rho1:+.3f}")
    plt.tight_layout(); plt.show()

    print(f"amplitude ratio (measured / claimed) = {k:.3f}")
    if abs(k - 1) < 0.05:
        print("  -> consistent: the map is a per-pixel sigma, add_noise injects the right amount.")
    elif k < 0.95:
        print(f"  -> the map is LARGER than the actual pixel scatter by 1/{k:.3f} = {1/k:.2f}x.")
        print("     Expected if the RMS map is inflated to absorb correlated noise: injecting")
        print("     WHITE noise at that sigma over-noises the reconstruction. Scale it by k,")
        print("     or generate noise with the measured correlation instead.")
    else:
        print("  -> the map is SMALLER than the actual scatter; check the column's convention.")
    if abs(k - np.sqrt(np.median(sigma_claimed[ok]))) < 0.05 * k:
        print("  !! ratio close to sqrt(map): the column may be a VARIANCE, not a sigma.")

    print(f"lag-1 autocorrelation = {rho1:+.3f}")
    if abs(rho1) < 0.05:
        print("  -> background is effectively white; add_noise's white-noise model is fine.")
    else:
        print("  -> background is spatially CORRELATED while add_noise injects white noise.")
        print("     Same marginal sigma, different power spectrum: segmap, Gini, A and M")
        print("     will differ between real and reconstruction for purely instrumental reasons.")
